In [ ]:
from datasets import load_dataset
from dotenv import load_dotenv
import os
import json
import datetime
import multiprocessing
import functools
from pathlib import Path
from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# LangChain imports
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_deepseek import ChatDeepSeek
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mistralai.chat_models import ChatMistralAI
from langchain_xai import ChatXAI
from langchain.schema.messages import SystemMessage, HumanMessage

# Configuration
class Config:
    # LLM Configuration
    LLM_PROVIDER = "claude"
    MODEL_NAME = "claude-3-5-sonnet-20241022"
    TEMPERATURE = 0.0
    MAX_TOKENS = 1000
    
    # System prompt
    SYSTEM_PROMPT = """You are an expert software engineer specializing in bug analysis. 
                       Your task is to determine the most likely file extensions related to bugs in code."""
    
    # Multiprocessing configuration
    NUM_PROCESSES = 1  # Adjusted for Linux environments which handle more processes efficiently
    
    # Analysis configuration
    NUM_SAMPLES = None  # None to process all samples, or an integer for a subset
    
    # Output structure 
    BASE_OUTPUT_DIR = "file_extensions"
    
    @classmethod
    def to_dict(cls):
        """Convert configuration to dictionary for saving"""
        return {
            "llm_provider": cls.LLM_PROVIDER,
            "model_name": cls.MODEL_NAME,
            "temperature": cls.TEMPERATURE,
            "max_tokens": cls.MAX_TOKENS,
            "system_prompt": cls.SYSTEM_PROMPT,
            "num_processes": cls.NUM_PROCESSES,
            "num_samples": cls.NUM_SAMPLES
        }

def create_experiment_dir():
    """Create an experiment directory with timestamp"""
    # Create base directory if it doesn't exist
    base_dir = Path(Config.BASE_OUTPUT_DIR)
    base_dir.mkdir(exist_ok=True)
    
    # Create experiment directory with timestamp
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    experiment_dir = base_dir / timestamp
    experiment_dir.mkdir(exist_ok=True)
    
    # Create subdirectories
    (experiment_dir / "raw_results").mkdir(exist_ok=True)
    
    # Save configuration
    with open(experiment_dir / "config.json", "w") as f:
        json.dump(Config.to_dict(), f, indent=2)
    
    return experiment_dir

def load_swe_bench_lite():
    """
    Load the SWE-bench Lite dataset using Hugging Face datasets library.
    
    Returns:
        Dataset: A Hugging Face Dataset containing the data.
    """
    dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test")
    return dataset

def load_llm(llm_provider: str, model_name: str, temperature: float, max_tokens: int):
    """Load the specified LLM model."""
    load_dotenv()  # Load environment variables from .env file
    
    if llm_provider == "chatgpt":
        return ChatOpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_provider == "claude":
        return ChatAnthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY"),
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_provider == "deepseek":
        return ChatDeepSeek(
            api_key=os.getenv("DEEPSEEK_API_KEY"),
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_provider == "gemini":
        return ChatGoogleGenerativeAI(
            api_key=os.getenv("GOOGLE_API_KEY"),
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_provider == "mistral":
        return ChatMistralAI(
            api_key=os.getenv("MISTRAL_API_KEY"),
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens
        )
    elif llm_provider == "grok":
        return ChatXAI(
            api_key=os.getenv("XAI_API_KEY"),
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens
        )
    else:
        raise ValueError(f"Unsupported LLM provider: {llm_provider}")

@retry(
    stop=stop_after_attempt(10),
    wait=wait_exponential(multiplier=1, min=30, max=300),
    retry=retry_if_exception_type((Exception))
)
def query_llm(prompt, llm_name, config, llm):
    """Query the specified LLM with retry logic."""
    try:
        # Build messages depending on whether the model supports system messages
        messages = []
        
        # Models that support system messages
        if llm_name in ["chatgpt", "claude", "gemini", "mistral", "grok"]:
            if config["system_prompt"]:
                messages.append(SystemMessage(content=config["system_prompt"]))
            messages.append(HumanMessage(content=prompt))
        else:
            # For models without system message support
            messages.append(HumanMessage(content=prompt))
        
        response = llm.invoke(messages)
        return response.content
    except Exception as e:
        print(f"Error querying {llm_name}: {str(e)}")
        raise

def extract_file_extensions(response):
    """
    Extract file extensions from the LLM response.
    
    Args:
        response: The raw response text from the LLM
        
    Returns:
        list: A list of extracted file extensions
    """
    extensions = []
    
    try:
        lines = response.strip().split('\n')
        for line in lines:
            # Look for numbered list items with file extensions
            if any(f"{i}. ." in line for i in range(1, 4)):
                # Extract just the extension part
                ext_part = line.split('.', 1)[1].strip()
                if ' ' in ext_part:
                    ext = ext_part.split(' ', 1)[0].strip()
                else:
                    ext = ext_part
                
                extensions.append(f".{ext}")
    except Exception as e:
        print(f"Error extracting extensions: {str(e)}")
    
    return extensions

def process_example(args):
    """
    Process a single example from the dataset.
    
    Args:
        args: Tuple containing (example, experiment_dir, llm_config, llm_provider, model_name, temperature, max_tokens)
    
    Returns:
        Tuple: (instance_id, extensions)
    """
    example, experiment_dir, llm_config, llm_provider, model_name, temperature, max_tokens = args
    
    # Load LLM inside the worker process
    llm = load_llm(llm_provider, model_name, temperature, max_tokens)
    
    instance_id = example.get("instance_id", "unknown")
    problem_statement = example.get("problem_statement", "")
    repo = example.get("repo", "")
    
    # Create the improved prompt
    prompt = f"""
    Analyze the following software bug description and determine the TOP 3 most likely file extensions associated with this issue in the {repo} repository.

    Problem Statement:
    {problem_statement}

    Based solely on the technical details in the problem statement, list the TOP 3 most likely file extensions where these bugs might be located.

    Format your response exactly as follows:
    1. [extension1]
    2. [extension2]
    3. [extension3]

    Your response must be exactly in this format with numbered points.
    Do not include any additional text, explanations, preambles, or postscripts.
    """
    
    try:
        # Query the LLM with retry
        response = query_llm(prompt, llm_provider, llm_config, llm)
        
        # Extract file extensions from the response
        extensions = extract_file_extensions(response)
        
        # Create result dictionaries
        raw_result = {
            "instance_id": instance_id,
            "problem_statement": problem_statement,
            "llm_response": response
        }
        
        # Save raw result to a separate file
        raw_result_path = os.path.join(experiment_dir, "raw_results", f"{instance_id}.json")
        
        # Use file system directly (no lock needed as each process writes to a different file)
        with open(raw_result_path, "w") as f:
            json.dump(raw_result, f, indent=2)
        
        # Return the extracted extensions
        return instance_id, extensions
    
    except Exception as e:
        print(f"Error processing problem with instance_id {instance_id}: {str(e)}")
        return instance_id, []

def normalize_extension(ext):
    """
    Normalize a file extension to ensure it has exactly one leading dot.
    
    Args:
        ext: The file extension string
        
    Returns:
        str: Normalized file extension with exactly one leading dot
    """
    # Strip all leading dots
    ext_stripped = ext.lstrip('.')
    
    # Add back exactly one dot
    return f".{ext_stripped}" if ext_stripped else ""

def validate_and_clean_extensions(results_dict):
    """
    Validate and clean the file extensions in the results dictionary.
    
    Args:
        results_dict: Dictionary with instance_ids as keys and extension lists as values
        
    Returns:
        dict: Dictionary with validated and cleaned extensions
    """
    cleaned_dict = {}
    
    for instance_id, extensions in results_dict.items():
        # Normalize each extension
        normalized_extensions = [normalize_extension(ext) for ext in extensions if ext]
        
        # Remove empty strings
        normalized_extensions = [ext for ext in normalized_extensions if ext]
        
        # Remove duplicates while preserving order
        unique_extensions = []
        for ext in normalized_extensions:
            if ext not in unique_extensions:
                unique_extensions.append(ext)
        
        # Store the cleaned extensions
        cleaned_dict[instance_id] = unique_extensions
    
    return cleaned_dict

def save_results(results_dict, experiment_dir):
    """Save the results dictionary to a file"""
    # Validate and clean extensions before saving
    cleaned_results = validate_and_clean_extensions(results_dict)
    
    with open(os.path.join(experiment_dir, "results.json"), "w") as f:
        json.dump(cleaned_results, f, indent=2)

# Create experiment directory
experiment_dir = create_experiment_dir()
print(f"Experiment directory created at: {experiment_dir}")

# Load the dataset
dataset = load_swe_bench_lite()

# Create config dictionary for LLM
llm_config = {
    "system_prompt": Config.SYSTEM_PROMPT
}

# Limit samples if specified
if Config.NUM_SAMPLES:
    dataset = dataset.select(range(min(Config.NUM_SAMPLES, len(dataset))))

# Convert dataset to list for multiprocessing
examples = list(dataset)

# Prepare arguments for worker processes
process_args = [
    (
        example, 
        str(experiment_dir),  # Convert Path to string to ensure it's picklable
        llm_config, 
        Config.LLM_PROVIDER, 
        Config.MODEL_NAME, 
        Config.TEMPERATURE, 
        Config.MAX_TOKENS
    ) 
    for example in examples
]

# Process examples in parallel
results_dict = {}

with multiprocessing.Pool(processes=Config.NUM_PROCESSES) as pool:
    # Process examples in parallel and collect results
    for instance_id, extensions in tqdm(
        pool.imap_unordered(process_example, process_args),
        total=len(examples),
        desc=f"Analyzing problems with {Config.NUM_PROCESSES} processes"
    ):
        results_dict[instance_id] = extensions
        
        # Periodically save the overall results (every 10 items)
        if len(results_dict) % 10 == 0:
            save_results(results_dict, experiment_dir)

# Save final results
save_results(results_dict, experiment_dir)

print(f"Analysis completed. Results saved to {experiment_dir}")